In [1]:
# !pip install qiskit

In [2]:
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import UnitaryGate

from qiskit.primitives import StatevectorSampler
import numpy as np

In [3]:
shots=10240
qubits=8 # use 10 with caution
marked_state = 0b111  # target state as integer (|111> = 7)

In [4]:
def measure(qc: QuantumCircuit):
    for i in range(qubits):
        qc.measure(i, i)

    sampler = StatevectorSampler()
    pm = generate_preset_pass_manager(optimization_level=1)

    isa = pm.run(qc)
    job = sampler.run([isa], shots=shots)
    pub = job.result()[0]
    return pub.data.c.get_counts()

def inversion_about_mean(qc: QuantumCircuit):
    # Inversion about mean circuit, derivation can be found in exercise sheet:
    # H^{⊗3} X^{⊗3} (X1 Z1 X1 Z1) CCZ_{1,2,3} X^{⊗3} H^{⊗3}

    qubits_list = list(range(qubits))

    qc.h(qubits_list)
    qc.x(qubits_list)

    qc.h(qubits-1)
    qc.mcx(list(range(qubits-1)), qubits-1)
    qc.h(qubits-1)
    # - sign change
    for _ in range(2):
        qc.z(0)
        qc.x(0)

    qc.barrier()
    qc.x(qubits_list)
    qc.h(qubits_list)

def oracle(qc: QuantumCircuit, n: int, marked: int):
    """General phase oracle for n qubits.
    Constructs the diagonal unitary  I - 2|marked><marked|
    which flips the phase of the marked state and leaves all others unchanged.
    """
    N = 2**(n+1)
    U = np.eye(N)
    # swap |marked,0> <-> |marked,1>  (ancilla is qubit n, most significant)
    idx0 = marked
    idx1 = 2**n + marked
    U[idx0, idx0] = 0
    U[idx1, idx1] = 0
    U[idx0, idx1] = 1
    U[idx1, idx0] = 1

    qc.barrier()
    qc.append(UnitaryGate(U, label=f"Oracle |{marked:0{n}b}>"), range(n+1))
    qc.barrier()

def construct_grover(qubits:int, k: int) -> QuantumCircuit:
    qc = QuantumCircuit(qubits+1, qubits)

    qc.x(qubits)
    
    for i in range(qubits+1):
        qc.h(i)

    for i in range(k):
        oracle(qc, n=qubits, marked=marked_state)
        inversion_about_mean(qc)
    return qc

In [5]:
# We test with applying Grover operator G k times and measure success probability
# for k in range(1, 11):
#     qc = construct_grover(qubits, k)
#     counts = measure(qc)

#     marked_bitstring = format(marked_state, f'0{qubits}b')
#     success_counts = counts.get(marked_bitstring, 0)
#     probability = success_counts / shots
#     theta = np.arcsin(np.sqrt(1/2**qubits))
#     prediction = np.sin((2*k+1)*theta)**2
#     print(f"k={k:2d}  success state probability: {probability:.5f}   theoretical prediction: {prediction:.5f}")


k_optimal = int(np.round(np.pi / 4 * np.sqrt(2**qubits) - 0.5))
print(f"optimal amount of repetitions: {k_optimal}")

qc = construct_grover(qubits, k_optimal)
counts = measure(qc)

marked_bitstring = format(marked_state, f'0{qubits}b')
success_counts = counts.get(marked_bitstring, 0)
probability = success_counts / shots
theta = np.arcsin(np.sqrt(1/2**qubits))
prediction = np.sin((2*k_optimal+1)*theta)**2
print(f"k={k_optimal:2d}  success state probability: {probability:.5f}   theoretical prediction: {prediction:.5f}")


optimal amount of repetitions: 12


k=12  success state probability: 0.99990   theoretical prediction: 0.99995


In [6]:
qc.draw(style="iqp")

┌───┐      ░ ┌────────────────────┐ ░ ┌───┐┌───┐          ┌───┐┌───┐┌───┐»
q_0: ┤ H ├──────░─┤0                   ├─░─┤ H ├┤ X ├───────■──┤ Z ├┤ X ├┤ Z ├»
     ├───┤      ░ │                    │ ░ ├───┤├───┤       │  └───┘└───┘└───┘»
q_1: ┤ H ├──────░─┤1                   ├─░─┤ H ├┤ X ├───────■─────────────────»
     ├───┤      ░ │                    │ ░ ├───┤├───┤       │                 »
q_2: ┤ H ├──────░─┤2                   ├─░─┤ H ├┤ X ├───────■─────────────────»
     ├───┤      ░ │                    │ ░ ├───┤├───┤       │                 »
q_3: ┤ H ├──────░─┤3                   ├─░─┤ H ├┤ X ├───────■─────────────────»
     ├───┤      ░ │                    │ ░ ├───┤├───┤       │                 »
q_4: ┤ H ├──────░─┤4 Oracle |00000111> ├─░─┤ H ├┤ X ├───────■─────────────────»
     ├───┤      ░ │                    │ ░ ├───┤├───┤       │                 »
q_5: ┤ H ├──────░─┤5                   ├─░─┤ H ├┤ X ├───────■─────────────────»
     ├───┤      ░ │                    │ ░ ├───┤├───┤       │                 »
q_6: ┤ H ├──────░─┤6                   ├─░─┤ H ├┤ X ├───────■─────────────────»
     ├───┤      ░ │                    │ ░ ├───┤├───┤┌───┐┌─┴─┐┌───┐          »
q_7: ┤ H ├──────░─┤7                   ├─░─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├──────────»
     ├───┤┌───┐ ░ │                    │ ░ └───┘└───┘└───┘└───┘└───┘          »
q_8: ┤ X ├┤ H ├─░─┤8                   ├─░────────────────────────────────────»
     └───┘└───┘ ░ └────────────────────┘ ░                                    »
c: 8/═════════════════════════════════════════════════════════════════════════»
                                                                              »
«     ┌───┐ ░ ┌───┐┌───┐ ░ ┌────────────────────┐ ░ ┌───┐┌───┐          ┌───┐»
«q_0: ┤ X ├─░─┤ X ├┤ H ├─░─┤0                   ├─░─┤ H ├┤ X ├───────■──┤ Z ├»
«     └───┘ ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤       │  └───┘»
«q_1: ──────░─┤ X ├┤ H ├─░─┤1                   ├─░─┤ H ├┤ X ├───────■───────»
«           ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤       │       »
«q_2: ──────░─┤ X ├┤ H ├─░─┤2                   ├─░─┤ H ├┤ X ├───────■───────»
«           ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤       │       »
«q_3: ──────░─┤ X ├┤ H ├─░─┤3                   ├─░─┤ H ├┤ X ├───────■───────»
«           ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤       │       »
«q_4: ──────░─┤ X ├┤ H ├─░─┤4 Oracle |00000111> ├─░─┤ H ├┤ X ├───────■───────»
«           ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤       │       »
«q_5: ──────░─┤ X ├┤ H ├─░─┤5                   ├─░─┤ H ├┤ X ├───────■───────»
«           ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤       │       »
«q_6: ──────░─┤ X ├┤ H ├─░─┤6                   ├─░─┤ H ├┤ X ├───────■───────»
«           ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤┌───┐┌─┴─┐┌───┐»
«q_7: ──────░─┤ X ├┤ H ├─░─┤7                   ├─░─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├»
«           ░ └───┘└───┘ ░ │                    │ ░ └───┘└───┘└───┘└───┘└───┘»
«q_8: ──────░────────────░─┤8                   ├─░──────────────────────────»
«           ░            ░ └────────────────────┘ ░                          »
«c: 8/═══════════════════════════════════════════════════════════════════════»
«                                                                            »
«     ┌───┐┌───┐┌───┐ ░ ┌───┐┌───┐ ░ ┌────────────────────┐ ░ ┌───┐┌───┐     »
«q_0: ┤ X ├┤ Z ├┤ X ├─░─┤ X ├┤ H ├─░─┤0                   ├─░─┤ H ├┤ X ├─────»
«     └───┘└───┘└───┘ ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤     »
«q_1: ────────────────░─┤ X ├┤ H ├─░─┤1                   ├─░─┤ H ├┤ X ├─────»
«                     ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤     »
«q_2: ────────────────░─┤ X ├┤ H ├─░─┤2                   ├─░─┤ H ├┤ X ├─────»
«                     ░ ├───┤├───┤ ░ │                    │ ░ ├───┤├───┤     »
«q_3: ────────────────░─┤ X ├┤ H ├─░─┤3                   ├─░─┤ H ├┤ X ├─────»
«                     ░ ├───┤├───┤